# Datamine DECILE Process Example

This notebook demonstrates how to configure and run the **`decile`** process wrapper in `dmstudio`.

## Process Description

## DECILE

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

Perform a decile analysis of a set of sample data.

**DECILE** retrieves samples above a supplied cut-off, they are sorted on grade, and then split into 10 bins, each representing one 'decile'. The average grade, maximum grade, minimum grade, total gold content and the proportion of the total gold content is then calculated for each bin. The top (90%+) bin is then further split into effective 1% bins and the calculations repeated. The results are stored in a system print file. The user has the option to add a topcut to the samples, and to use simple sample record selection if desired. Please see the related process [QUANTILE](<quantile.md>).

### Input Files:

* **in** (Undefined):
  Input sample file
  Required=Yes

### Output Files:

* **priout** (Undefined):
  Output print file containing sample decile information.
  Required=Yes

* **decout** (Table File):
  Output file containing sample decile information. Not including the top 10% subsplit
  Required=No

* **splitout** (Table File):
  Output file containing sample decile information. Including the top 10% subsplit
  Required=No

### Fields:

* **value** (Numeric : IN):
  Name of the field containing the grade to be processed.
  Default=Undefined
  Required=Yes

* **select** (Numeric : IN):
  Field containing the parameter for sample record selection.
  Default=Undefined
  Required=No

### Parameters:

* **cutoff**:
  Cutoff grade.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **topcut**:
  Apply a topcut to the file: 0 = No topcut applied. 1 = Topcut applied at grade defined
  in TCUTOFF
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **tcutoff**:
  Grade to be applied as a topcut, if TOPCUT is set to 1.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **criteria**:
  Value to be used in sample record selection - SELECT. eg. If ROCK is the field entered
  in SELECT and 2 is the value entered in CRITERIA, then the Decile analysis will only
  take place on records where ROCK = 2. This value must be numeric.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('decile')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute decile
print("Running decile...")
dm_cmd.decile(
    in_i='t_assays',  # required
    priout_o='t_decile_out',  # required
    decout_o='t_decile_out',  # required
    splitout_o='t_decile_out',  # required
    value_f='optional',  # required
    # select_f='optional',  # optional
    # cutoff_p='optional',  # optional
    # topcut_p=0,  # optional
    # tcutoff_p='optional',  # optional
    # criteria_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("decile execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_decile_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")